# Dag 10: AI Search + RAG + Responsible AI

**Eksamensrelevans:** Optimize Language Models for AI Applications + Design and Prepare a ML Solution — samlet ~45-55% af DP-100 eksamen

**Vigtig note:** Eksamen er IMORGEN. Denne notebook er et intensivt eksamenscrash-kursus. Emnerne her optræder primært som **konceptuelle/vidensbaserede** spørgsmål — fokus er forståelse, distinktioner og genkendelse af begreber.

**Nøglebegreber:**
- **Azure AI Search**: index-struktur, feltattributter, søge-typer (keyword, semantic, vector, hybrid)
- **Skillsets og enrichment**: built-in skills, custom skills, indexers
- **RAG-pattern**: retrieve → augment → generate, chunking, embeddings, "On Your Data"
- **Responsible AI**: de 6 principper, RAI dashboard-komponenter, Content Safety
- **Model Interpretability**: feature importance, SHAP, ErrorAnalysis, Counterfactuals, Causal

**Læringsmål:**
- Beskrive Azure AI Search-arkitekturen og hvilke feltattributter der gælder for hvilke operationer
- Skelne mellem keyword-, semantic-, vector- og hybrid-søgning og vide hvornår hvad bruges
- Forklare RAG-patternets tre trin og de centrale designvalg
- Navngive og beskrive de 6 Responsible AI-principper
- Kende de fire RAI dashboard-komponenter og hvad de måler

**Forudsætninger:** Dag 1-9 gennemført. Dag 8+9 (AI Foundry + Prompt Flow) er særligt relevant som kontekst.

---
## DEL 1: Azure AI Search
---

## 1. Hvad er Azure AI Search?

**Azure AI Search** (tidligere kaldet Azure Cognitive Search) er Microsofts fuldt administrerede søgetjeneste, der understøtter fuld-tekst søgning, semantisk søgning og vektorsøgning over store mængder dokumenter.

### Hvornår bruger du Azure AI Search?
- Du har en stor dokumentsamling (PDF'er, Word-filer, webindhold, databaser) og vil gøre dem søgbare
- Du bygger en **RAG-applikation** der skal hente relevant kontekst til en LLM
- Du har brug for **avanceret søgning** med filtre, facets, auto-complete og relevansranking
- Du vil **berige** dokumenter med AI (f.eks. uddrag nøglefraser, oversætte tekst, identificere entities)

### Arkitekturens hoveddele

```
Datakilde (Blob, SQL, Cosmos DB, ...)
         ↓  [Indexer]
    [Skillset]  ← AI-berigelse (OCR, oversættelse, key phrases, ...)
         ↓
      [Index]  ← søgbart lager
         ↑
    Søgeapplikation / LLM (RAG)
```

> **Eksamenstip:** De tre centrale komponenter at kende er **Index**, **Indexer** og **Skillset**. En **Datakilde** (data source) er en fjerde komponent — den definerer *hvorfra* data hentes. Disse fire udgør den fulde indexeringspipeline.

## 2. Index-struktur: Felter og attributter

Et **index** i Azure AI Search er analogt til en tabel i en database — det indeholder **felter** (kolonner) med definerede typer og **attributter** (egenskaber).

### Felttyper

| Type | Beskrivelse | Eksempel |
|---|---|---|
| `Edm.String` | Tekst | Titel, indhold, kategori |
| `Edm.Int32 / Int64` | Heltal | Sidetal, antal |
| `Edm.Double` | Decimaltal | Pris, score |
| `Edm.DateTimeOffset` | Dato/tid | Publiceringsdato |
| `Edm.Boolean` | Sand/falsk | Er aktiv |
| `Collection(Edm.String)` | Liste af strenge | Tags, forfattere |
| `Collection(Edm.Single)` | Vektor (embeddings) | Semantisk repræsentation |

### Feltattributter — den vigtigste tabel til eksamen

| Attribut | Hvad gør det? | Nødvendigt for |
|---|---|---|
| **searchable** | Feltet analyseres og indekseres til fuld-tekst søgning | Søgning i tekstindhold |
| **filterable** | Feltet kan bruges i `$filter`-udtryk | `filter=category eq 'Azure'` |
| **sortable** | Feltet kan bruges til at sortere resultater | `orderby=date desc` |
| **facetable** | Feltet kan bruges til at lave facetaggregering (grupper) | Facet-navigation ("Vis 42 resultater for 'Azure'") |
| **retrievable** | Feltet returneres i søgeresultater | Vise titel og indhold i UI |
| **key** | Unikt dokument-id — ét felt per index | Identifikation af dokumenter |
| **hidden** | Feltet gemmes, men returneres ikke (overskriver retrievable) | Interne beregninger |

> **Eksamenstip:** Typisk eksamensscenarie: *"Du vil filtrere søgeresultater på kategori, men kategorien skal ikke søges i. Hvilke attributter sætter du på kategori-feltet?"* — Svar: `filterable=true`, `searchable=false`. Disse attributter er **uafhængige** — du kan have et felt der er filterable men ikke searchable, og omvendt.

> **Eksamenstip:** `facetable` og `sortable` virker **kun** på felter med ikke-teksttyper eller enkeltværdi-strenge — de virker **ikke** på felter af typen `Collection(Edm.String)` eller analyserede tekstfelter.

## 3. Søgetyper: Keyword, Semantic, Vector og Hybrid

Azure AI Search understøtter fire overordnede søgemodi. At kende dem og hvornår de bruges er centralt til eksamen.

### Keyword Search (fuld-tekst)
- **Hvordan:** Klassisk fuld-tekst søgning med TF-IDF baseret relevansranking (BM25)
- **Hvad den finder:** Dokumenter der indeholder de eksakte ord fra søgeforespørgslen
- **Ulempe:** Forstår ikke *mening* — søger på "bil" finder ikke dokumenter om "automobil"
- **Hvornår:** Standard søgning, høj kontrol over resultater, ingen AI-afhængighed

### Semantic Search (semantisk re-ranking)
- **Hvordan:** Bruger Microsofts sprogmodel til at **re-ranke** keyword-søgeresultater baseret på *mening*
- **Hvad den finder:** Samme dokumenter som keyword, men i bedre rækkefølge + udtrækker **semantiske svar** og **captions**
- **Krav:** Kræver `semantic` konfiguration på index + `Standard` eller højere SKU
- **Hvornår:** Du vil have bedre relevans uden at bygge din egen embedding-model

### Vector Search
- **Hvordan:** Konverterer tekst til **embedding-vektorer** og finder dokumenter med lignende vektorer (cosine similarity)
- **Hvad den finder:** Semantisk lignende dokumenter, selv uden orddeling
- **Krav:** Du skal selv generere embeddings (f.eks. via Azure OpenAI `text-embedding-ada-002`) og gemme dem i et `Collection(Edm.Single)`-felt
- **Hvornår:** RAG-applikationer, find-lignende-dokumenter, flersproget søgning

### Hybrid Search
- **Hvordan:** Kombinerer keyword og vector søgning med **Reciprocal Rank Fusion (RRF)** for at fusionere rankinglister
- **Hvad den finder:** Dokumenter der matcher både præcist og semantisk — bedste af begge verdener
- **Hvornår:** RAG-applikationer i produktion — typisk bedst overordnet resultat

```
Hybrid Search = Keyword (BM25) + Vector (cosine) → RRF Fusion → (optionelt) Semantic Re-ranking
```

| Søgetype | Forstår mening? | Kræver embeddings? | Kræver Semantic config? | Bedst til |
|---|---|---|---|---|
| Keyword | Nej | Nej | Nej | Præcis match, lav latency |
| Semantic | Delvist | Nej | Ja | Bedre ranking af keyword-resultater |
| Vector | Ja | Ja | Nej | Semantisk lighed, RAG |
| Hybrid | Ja | Ja | Valgfrit | RAG i produktion |

> **Eksamenstip:** **Hybrid Search med Semantic Re-ranking** er best practice til RAG i produktion ifølge Microsoft. Semantic Search *re-ranker* eksisterende resultater — den finder ikke nye dokumenter. Vector Search finder nye dokumenter baseret på mening.

## 4. Skillsets og AI-berigelse

Et **Skillset** er en pipeline af **skills** der transformerer og beriger dokumenter under indexeringen. Skills anvendes på raw-dokumenter og tilføjer nye felter til index'et.

### Built-in Skills (AI-drevne)

| Skill | Hvad gør den? | Eksempel output |
|---|---|---|
| **OCR** | Læser tekst fra billeder i dokumenter | `text: "Azure AI Search..."` |
| **Language Detection** | Identificerer dokumentets sprog | `languageCode: "da"` |
| **Key Phrase Extraction** | Uddrager vigtige nøglefraser | `keyPhrases: ["Azure AI", "vektor søgning"]` |
| **Entity Recognition** | Genkender navne, steder, organisationer | `entities: [{text: "Microsoft", type: "Organization"}]` |
| **Text Translation** | Oversætter tekst til et målsprog | `translatedText: "Azure AI Search..."` |
| **Sentiment Analysis** | Måler positiv/negativ tone | `sentiment: "positive", score: 0.92` |
| **Image Analysis** | Beskriver billeder med tekst | `description: "En graf der viser..."` |
| **Text Split** | Opdeler lange tekster i chunks | `textItems: ["chunk1", "chunk2"]` |
| **Azure OpenAI Embedding** | Genererer embedding-vektorer | `embedding: [0.02, -0.13, ...]` |

### Custom Skills
- Du kan bygge **dine egne skills** som Web API-endpoints (typisk Azure Functions)
- Modtager dokumentdata som JSON, returnerer berignede data som JSON
- Bruges til domænespecifik logik der ikke er dækket af built-in skills

### Indexers
En **Indexer** orkesterer hele processen:
1. Henter dokumenter fra **datakilden** (Azure Blob, Azure SQL, Cosmos DB, SharePoint, m.fl.)
2. Sender dokumenter igennem **Skillset** (hvis konfigureret)
3. Kortlægger felter og gemmer resultater i **Index**
4. Kan køre **on-demand** eller efter en **tidsplan** (f.eks. hvert 5. minut)

> **Eksamenstip:** Skelnen built-in vs. custom skills: *Built-in skills* er Microsoft-leverede og bruges direkte uden kode. *Custom skills* kræver at du bygger et Web API (typisk Azure Function) der modtager og returnerer JSON i det specificerede format — men du styrer al logikken.

> **Eksamenstip:** En **Indexer** er ikke det samme som et **Index**. Indexeren er arbejderen der *udfylder* index'et — index'et er *lageret* der søges i. Du kan have et index uden en indexer (manuel upload via REST API).

## 5. Kodeøvelse: Gennemgå et AI Search index-schema

I denne øvelse bruger du `azure-search-documents` SDK'et til at hente metadata om et eksisterende search index og forstå dets feltattributter.

**Opgave:** Opret forbindelse til et Azure AI Search-index og list alle felter med deres attributter.

*Hint:* Du skal bruge `SearchIndexClient` fra `azure.search.documents.indexes`.

*Hint:* Brug `os.getenv("AZURE_SEARCH_ENDPOINT")` og `os.getenv("AZURE_SEARCH_KEY")` til credentials. Disse skal ligge i din `.env`-fil.

*Hint:* `client.get_index(index_name).fields` returnerer en liste af `SearchField`-objekter med `.name` og `.searchable`, `.filterable`, `.sortable`, osv.

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv("../.env")

# TODO: Importer SearchIndexClient og AzureKeyCredential
# HINT: from azure.search.documents.indexes import SearchIndexClient
# HINT: from azure.core.credentials import AzureKeyCredential

# TODO: Opret en SearchIndexClient med endpoint og key fra environment variables
# HINT: SearchIndexClient(endpoint=..., credential=AzureKeyCredential(...))
index_client = ...

# TODO: Hent et specifikt index (sæt index-navn herunder) og list alle felter
INDEX_NAME = "dit-index-navn"  # skift til dit eget index-navn

# TODO: Iterér over index.fields og print: name, type, searchable, filterable, sortable, facetable
# HINT: Hvert field-objekt har disse attributter som boolean-værdier
print(f"Felter i index '{INDEX_NAME}':")
# din kode her

---
## DEL 2: RAG (Retrieval-Augmented Generation)
---

## 6. RAG-pattern: Retrieve → Augment → Generate

**RAG (Retrieval-Augmented Generation)** er et arkitekturmønster der kombinerer en søgemaskine med en generativ AI-model (LLM) for at svare på spørgsmål baseret på din *egne* data — ikke kun LLM'ens træningsdata.

### De tre trin

```
Brugerspørgsmål: "Hvad er vores returpolitik?"

1. RETRIEVE
   └── Søg i Azure AI Search index med spørgsmålet
   └── Hent de N mest relevante dokumentchunks
   └── Resultat: ["Vores returpolitik er...", "Varer kan returneres inden..."]

2. AUGMENT
   └── Byg en prompt der inkluderer de retrievede chunks som kontekst
   └── Prompt: "Besvar spørgsmålet KUN baseret på denne kontekst: [chunks]\n\nSpørgsmål: ..."

3. GENERATE
   └── Send den augmenterede prompt til LLM (f.eks. GPT-4o)
   └── LLM genererer svar baseret KUN på konteksten
   └── Resultat: "Ifølge vores politik kan du returnere..." + kildehenvisninger
```

### Fordele ved RAG vs. Fine-tuning

| | RAG | Fine-tuning |
|---|---|---|
| **Dataopdatering** | Realtid (opdater index) | Kræver ny træning |
| **Omkostning** | Lavere (kun søgning + inference) | Højere (GPU-træning) |
| **Transparens** | Kan citere kilder | Svar er "bagt ind" i model |
| **Hallucination** | Reduceret (bundet til kontekst) | Stadig risiko |
| **Egnet til** | Dynamisk, dokumentbaseret viden | Ændring af adfærd/stil/format |

> **Eksamenstip:** Eksamen vil typisk spørge *"Hvornår bruger du RAG vs. fine-tuning?"*. Huskeregel: **RAG** til dynamisk data og faktabaserede svar. **Fine-tuning** til at ændre modellens *adfærd, tone eller output-format* — ikke til at tilføje ny viden.

## 7. Chunking og Embeddings

### Chunking-strategier

LLMs har en **kontekstvindue-grænse** — du kan ikke sende hele dokumenter som kontekst. Chunking opdeler dokumenter i mindre stykker til indexering.

| Strategi | Beskrivelse | Egnet til |
|---|---|---|
| **Fixed-size** | Opdel i chunks af N tokens med overlap | Simple dokumenter, hurtig implementering |
| **Sentence-based** | Opdel ved sætningsgrænser | Tekst der skal læses sammenhængende |
| **Paragraph/section-based** | Opdel ved naturlige afsnitsgrænser | Strukturerede dokumenter (Word, PDF) |
| **Semantic chunking** | Opdel baseret på emne-skift (embedding-afstand) | Komplekse dokumenter med blandede emner |

**Overlap:** Chunks bør have *overlap* (f.eks. 10-20% af chunk-størrelse) for at undgå at vigtig kontekst splittes midt over en grænse.

> **Eksamenstip:** Azure AI Search har en built-in **Text Split skill** i skillsets til at opdele dokumenter. I RAG-pipelines er chunk-størrelsen et **hyperparameter** der påvirker retrieval-kvaliteten: For store chunks → irrelevant støj, for små chunks → manglende kontekst.

### Embeddings til Vector Search

**Embeddings** er numeriske repræsentationer af tekst som vektorer i et højdimensionalt rum. Semantisk lignende tekst har vektorer der er tæt på hinanden.

**Embedding-processen i RAG:**
1. Under **indexering**: Brug embedding-model til at konvertere hvert chunk → vektor → gem i index
2. Under **søgning**: Konvertér brugerens spørgsmål → vektor → find nærmeste naboer i index

**Anbefalede embedding-modeller i Azure:**
- `text-embedding-ada-002` (Azure OpenAI) — 1536 dimensioner, god generel performance
- `text-embedding-3-small` / `text-embedding-3-large` (Azure OpenAI) — nyere, mere effektive

> **Eksamenstip:** Den **samme embedding-model** skal bruges til at generere vektorer under indexering OG under søgning. Blander du modeller, bliver sammenligningerne meningsløse. Dette er en hyppig eksamens-fælde.

## 8. "On Your Data" — Azure OpenAI integration

**"On Your Data"** (OYD) er en Azure OpenAI-feature der bygger RAG-integrationslaget direkte ind i Azure OpenAI Service — uden at du selv skal skrive retrieve-og-augment koden.

### Hvad det gør
- Du konfigurerer Azure OpenAI til at kende dit **Azure AI Search index**
- Når du kalder Azure OpenAI API'et, henter det *automatisk* relevant kontekst fra dit index
- Modellen svarer baseret på din data og inkluderer **kildehenvisninger**
- Implementeret via `data_sources`-parameteren i API-kaldet

### Arkitektur

```
Bruger → Azure OpenAI API-kald (med data_sources=[azure_search_config])
                    ↓
        Azure OpenAI henter kontekst fra AI Search
                    ↓
        GPT-4o genererer svar med kildehenvisninger
                    ↓
        { answer: "...", citations: [{title, url, content}] }
```

### Alternativ: Byg selv i Prompt Flow
- Prompt Flow giver mere kontrol — du definerer hvert trin (retrieve, format, generate)
- OYD er hurtigere at komme i gang med, men mindre fleksibel

> **Eksamenstip:** *"On Your Data"* er den **enkleste** måde at bygge RAG på med Azure-tjenester, fordi Azure OpenAI håndterer retrieval-trinnet automatisk. Prompt Flow giver mere kontrol og fleksibilitet men kræver mere konfiguration.

## 9. Refleksion: AI Search + RAG

Svar på spørgsmålene herunder som kommentarer eller i en ny markdown-celle.

**Spørgsmål A:** Du indekserer en stor samling af PDF-fakturaer og skal tillade søgning på beløb (f.eks. `beløb > 5000`), men beløbet skal ikke søges i som fritekst. Hvilke feltattributter bruger du på beløb-feltet?

**Spørgsmål B:** Du bygger en RAG-applikation der skal finde dokumenter om *koncepter*, selv når brugeren bruger synonymer og omformuleringer. Hvilken søgetype er mest egnet, og hvad kræver det i index-strukturen?

**Spørgsmål C:** Din organisation vil give en chatbot adgang til jeres interne dokumentation (500 PDF-filer der opdateres ugentligt). Vil du bruge RAG eller fine-tuning? Begrund dit valg.

**Spørgsmål D:** Du har brugt to forskellige embedding-modeller — `text-embedding-ada-002` til indexering og `text-embedding-3-small` til søgning. Hvad er konsekvensen, og hvad skal du gøre for at rette det?

**Svar:**

A. 

B. 

C. 

D. 

---
## DEL 3: Responsible AI
---

## 10. De 6 Responsible AI-principper

Microsoft har defineret seks principper for Responsible AI. Disse er eksamensrelevante — du skal kende dem og kunne genkende eksempler.

| Princip | Beskrivelse | Eksempel på brud |
|---|---|---|
| **Fairness** | AI-systemer skal behandle alle mennesker retfærdigt og undgå bias | Låneansøgningsmodel der afviser bestemte befolkningsgrupper uforholdsmæssigt |
| **Reliability & Safety** | AI-systemer skal fungere pålideligt og sikkert under normale og uventede forhold | Selvkørende bil der fejler i regn fordi træningsdata kun indeholdt tørvejr |
| **Privacy & Security** | AI-systemer skal respektere privatliv og beskytte data | Model der memorerer og lækker persondata fra træningssættet |
| **Inclusiveness** | AI-systemer skal gavne og engagere alle mennesker | Talegenkendelsessystem der kun fungerer godt for bestemte accenter |
| **Transparency** | AI-systemer skal være forklarbare — brugere skal forstå, hvad de gør | "Black box"-kreditvurderingsmodel der ikke kan forklare afslagsårsager |
| **Accountability** | Mennesker skal stå til ansvar for AI-systemer | Ingen klar ansvarsfordeling ved AI-fejl i medicinsk diagnose |

> **Eksamenstip:** Disse seks principper er en fast del af DP-100. Exam-spørgsmål giver typisk et scenarie og beder dig identificere *hvilket princip der krænkes*. Huskeregel:
> - **Fairness** = bias mod grupper
> - **Transparency** = kan ikke forklare beslutninger
> - **Inclusiveness** = ekskluderer brugergrupper
> - **Privacy** = datalækage, memorering
> - **Reliability** = fejler i edge cases
> - **Accountability** = ingen ansvarlig person/team

## 11. Responsible AI Dashboard i Azure ML

**Azure ML's RAI Dashboard** er et interaktivt værktøj der samler fire analysekomponenter til at vurdere modellers opførsel og identificere problemer.

### De fire RAI Dashboard-komponenter

#### 1. Error Analysis
- **Hvad:** Identificerer hvilke **datakohorteer** (subgrupper) modellen fejler på
- **Visualisering:** Et fejltræ (decision tree) der viser fejlkoncentrationer + en heatmap
- **Spørger:** *"Fejler min model mere for ældre brugere? For lavindkomst-grupper?"*
- **RAI-princip:** Fairness, Reliability

#### 2. Explanations (Model Interpretability)
- **Hvad:** Forklarer **hvilke features** der driver modelens forudsigelser (globalt og per observation)
- **Metode:** SHAP (SHapley Additive exPlanations) — matematisk korrekt feature importance
- **Spørger:** *"Hvorfor forudsagde modellen 'høj risiko' for denne kunde?"*
- **RAI-princip:** Transparency

#### 3. Counterfactuals
- **Hvad:** Finder den **minimale ændring** i input-features der ville have givet et andet udfald
- **Eksempel:** *"Hvad skulle ændres for at låneansøgningen var gået igennem?"* → "Indkomst +5.000 kr/md"
- **Spørger:** *"Hvad skal ændres for at opnå et andet resultat?"*
- **RAI-princip:** Transparency, Fairness

#### 4. Causal Analysis
- **Hvad:** Estimerer den **kausale effekt** af at ændre en feature på udfaldet (ikke kun korrelation)
- **Eksempel:** *"Hvis vi tilbyder rabat, øger det faktisk sandsynligheden for køb?"*
- **Spørger:** *"Hvad sker der HVIS vi ændrer denne variabel?"*
- **RAI-princip:** Accountability, Reliability

```
RAI Dashboard
├── Error Analysis     → Hvilke grupper fejler modellen på?
├── Explanations       → Hvilke features driver forudsigelserne? (SHAP)
├── Counterfactuals    → Hvad skal ændres for et andet udfald?
└── Causal Analysis    → Hvad er den kausale effekt af en feature?
```

> **Eksamenstip:** Disse fire komponenter er en hyppig eksamenstest. Nøgle-distinktionen:
> - **Error Analysis** = *find* fejlkoncentrationer (fejltræ)
> - **Explanations/SHAP** = *forstå* feature-indflydelse
> - **Counterfactuals** = *hvad skal ændres* for et andet udfald
> - **Causal Analysis** = *kausale effekter* (interventioner)

## 12. Model Interpretability: Feature Importance og SHAP

### Global vs. Lokal Forklaring

| Type | Beskrivelse | Eksempel |
|---|---|---|
| **Global feature importance** | Hvilke features er vigtigst for *alle* forudsigelser samlet? | "Alder" er den vigtigste feature for churns-modellen generelt |
| **Lokal feature importance** | Hvilke features drev forudsigelsen for *én specifik observation*? | "Manglende lønstigninger" drev *denne* kundes churn-forudsigelse |

### SHAP (SHapley Additive exPlanations)
SHAP er en spilleteori-baseret metode der beregner hvert features **marginale bidrag** til forudsigelsen.

**SHAP-værdier:** Et positivt SHAP-tal for en feature betyder at den *øger* forudsigelsen. Negativt SHAP = *reducerer* forudsigelsen.

**Visualiseringstyper:**
- **Beeswarm plot**: Viser alle observationer — hvert punkt er én observation, farven indikerer feature-værdien
- **Bar plot**: Gennemsnitlig absolut SHAP-værdi per feature (global importance)
- **Waterfall plot**: Én observations forudsigelse nedbrudt i feature-bidrag (lokal forklaring)

### Feature Importance i Azure ML
Azure ML beregner feature importance via:
- **RAI Dashboard Explanations-komponent** (anbefalet, SHAP-baseret)
- **AutoML**: Giver automatisk feature importance for det bedste model
- **MLflow**: Du kan logge SHAP-plots som artifacts med `mlflow.log_artifact()`

> **Eksamenstip:** *"Hvilken metode bruger Azure ML's RAI Dashboard til at beregne feature importance?"* — Svar: **SHAP** (SHapley Additive exPlanations). Det er den eneste metode der er *guaranteed* consistent og korrekt for alle modeltyper.

## 13. Content Safety i Azure AI

**Azure AI Content Safety** er en tjeneste der screener tekst og billeder for skadeligt indhold — som en sikkerhedsbarriere foran eller bag en LLM.

### Hvad den detekterer

| Kategori | Beskrivelse |
|---|---|
| **Hate** | Hadtale, diskriminerende indhold |
| **Violence** | Voldsomt eller truende indhold |
| **Sexual** | Seksuelt eksplicit indhold |
| **Self-harm** | Indhold om selvskade eller selvmord |

Hvert kategori returnerer en **alvorligheds-score** (0-7) og en **handling** (allow/block).

### Prompt Shields (Jailbreak Detection)
- Detekterer **jailbreak-forsøg** — brugerinput der prøver at omgå modellens sikkerhedsrestriktioner
- Detekterer **indirect prompt injection** — skjulte instruktioner i dokumenter der forsøger at manipulere LLM'en

### Integrationspoint
- **Input-screening**: Screen brugerens prompt *før* den sendes til LLM
- **Output-screening**: Screen LLM'ens svar *før* det vises til brugeren
- Tilgængeligt via REST API eller Python SDK (`azure-ai-contentsafety`)

> **Eksamenstip:** Content Safety er *ikke* en del af RAI Dashboard — det er en **separat Azure AI-tjeneste**. RAI Dashboard er til model-analyse (fairness, explainability). Content Safety er til **runtime-beskyttelse** af LLM-applikationer.

## 14. Kodeøvelse: Tjek RAI Dashboard komponenter via SDK

RAI Dashboard bygges via en Azure ML **pipeline** med specielle RAI-komponenter. I denne øvelse kigger du på, hvad der kræves for at konstruere en RAI-analyse-pipeline.

**Opgave:** List de RAI-komponenter der er tilgængelige i dit workspace via registry.

*Hint:* RAI Dashboard-komponenterne er registreret i Azure ML's **AzureML registry** — ikke i dit workspace direkte. Brug `MLClient` med registry-navn `"azureml"` til at browse dem.

*Hint:* `ml_client_registry = MLClient(credential=DefaultAzureCredential(), registry_name="azureml")` og derefter `ml_client_registry.components.list(name="microsoft_azureml_rai_...")` for at finde RAI-komponenter.

*Hint:* RAI-komponenter starter typisk med `rai_` i deres navn, f.eks. `rai_insights_dashboard`, `rai_explanation_component`.

In [ ]:
import sys
sys.path.append("..")

from src.utils import init_ml_client
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

ml_client = init_ml_client()

# TODO: Opret en MLClient der peger på AzureML-registret (registry_name="azureml")
# HINT: MLClient(credential=DefaultAzureCredential(), registry_name="azureml")
ml_client_registry = ...

# TODO: List komponenter fra registret der indeholder "rai" i navnet
# HINT: ml_client_registry.components.list() — iterér og filtrer på .name.startswith("rai")
print("RAI-komponenter i AzureML registry:")
# din kode her

## 15. Refleksion: Responsible AI

Svar på spørgsmålene herunder. Dette er typiske DP-100-spørgsmål om Responsible AI.

**Spørgsmål A:** Din kreditvurderingsmodel afviser låneansøgninger fra en bestemt postnummerzone markant oftere end andre zoner. Hvilket RAI-princip er krænket, og hvilken RAI Dashboard-komponent ville du starte med for at undersøge problemet?

**Spørgsmål B:** En kunde spørger: *"Hvorfor fik jeg afslag på min låneansøgning, og hvad skulle jeg ændre for at blive godkendt?"* Hvilke to RAI Dashboard-komponenter er relevante for at besvare disse to spørgsmål?

**Spørgsmål C:** Du vil tjekke om brugere af din chatbot forsøger at manipulere LLM'en via skjulte instruktioner i de dokumenter de uploader. Hvilken Azure-tjeneste og feature bruger du?

**Spørgsmål D:** Hvad er forskellen på **global feature importance** og **lokal feature importance**? Giv et konkret eksempel på hvornår du bruger hver.

**Svar:**

A. 

B. 

C. 

D. 

---
## 16. Samlet Eksamensquiz: Dag 10 — AI Search + RAG + Responsible AI
---

Dette er den vigtigste sektion. Svar på alle spørgsmål — de efterligner DP-100 eksamensformat. Svar er til sidst i sektionen.

---

**Spørgsmål 1 (Feltattributter):**
Du bygger et product-search index. Feltet `price` (Edm.Double) skal bruges til at filtrere produkter (`price < 1000`) og sortere resultater (`orderby=price asc`), men brugere skal ikke kunne søge fritekst i prisfeltet. Hvilken kombination af attributter sætter du på `price`?

A) `searchable=true, filterable=true, sortable=true`
B) `searchable=false, filterable=true, sortable=true`
C) `searchable=true, filterable=false, sortable=true`
D) `searchable=false, filterable=false, sortable=true`

---

**Spørgsmål 2 (Søgetyper):**
Du vil bygge en RAG-søgning der finder dokumenter der er *semantisk relevante*, selv om de ikke indeholder de eksakte ord fra brugerens spørgsmål. Hvilken søgetype er nødvendig, og hvad kræver det af dit index?

A) Semantic Search — kræver `semantic`-konfiguration på index med Standard-SKU
B) Vector Search — kræver et `Collection(Edm.Single)`-felt med embedding-vektorer i index
C) Keyword Search — standard fuld-tekst søgning understøtter semantisk lighed
D) Hybrid Search — kræver både `semantic`-konfiguration og `Collection(Edm.Single)`-felt

---

**Spørgsmål 3 (RAG vs. Fine-tuning):**
Din organisation vil gøre 10.000 interne tekniske manualer søgbare via en chatbot. Manualerne opdateres månedligt. Hvilken tilgang anbefaler du?

A) Fine-tune GPT-4o på manualerne månedligt, da modellen derved kender dokumenterne fuldt ud
B) RAG med Azure AI Search + Azure OpenAI, fordi indeksopdatering er billigere end månedlig fine-tuning
C) RAG er ikke egnet til tekniske manualer — brug i stedet keyword search direkte mod PDF-filer
D) Fine-tune en Phi-3 model én gang — den vil automatisk lære de nye manualer ved re-inference

---

**Spørgsmål 4 (Skillsets):**
Du indexerer en samling scannede papirdokumenter (billeder af tekst). Hvilken built-in skill er nødvendig for overhovedet at kunne søge i indholdet?

A) Entity Recognition — identificerer tekst-entiteter i billeder
B) Image Analysis — beskriver billeder med tekst
C) OCR (Optical Character Recognition) — læser tekst fra billedfiler
D) Text Translation — konverterer billedindhold til søgbar tekst

---

**Spørgsmål 5 (Embeddings):**
Du indexerede 50.000 dokumenter med `text-embedding-ada-002` (1536 dimensioner). Du skifter nu til `text-embedding-3-small` (1536 dimensioner) til søgning for at reducere latency. Hvad er konsekvensen?

A) Ingen konsekvens — begge modeller har samme antal dimensioner, vektorerne er kompatible
B) Søgeresultaterne bliver meningsløse fordi de to modeller producerer inkompatible vektorrum
C) Performance forbedres automatisk da `text-embedding-3-small` er mere effektiv
D) Azure AI Search konverterer automatisk vektorerne til det nye format

---

**Spørgsmål 6 (RAI Dashboard):**
Din churn-forudsigelsesmodel har høj samlet nøjagtighed, men du mistænker at den performer dårligt for kunder med lav anciennitet. Hvilken RAI Dashboard-komponent bruger du til at identificere dette?

A) Explanations — viser feature importance for anciennitet
B) Counterfactuals — viser hvad der skal ændres for at ændre forudsigelsen
C) Error Analysis — viser fejlkoncentrationer i datakohorteer via fejltræ
D) Causal Analysis — viser den kausale effekt af anciennitet på churn

---

**Spørgsmål 7 (RAI-principper):**
Et AI-system til automatisk billedgenkendelse fungerer med 95% nøjagtighed for lys hud, men kun 72% for mørk hud, fordi træningsdatasættet var ubalanceret. Hvilket Responsible AI-princip er *primært* krænket?

A) Reliability & Safety — fordi systemet ikke er pålideligt
B) Fairness — fordi systemet behandler brugergrupper uretfærdigt baseret på hudfarve
C) Inclusiveness — fordi systemet ikke er tilgængeligt for alle brugere
D) Transparency — fordi modellen ikke kan forklare sine fejl

---

**Spørgsmål 8 (Content Safety):**
Din RAG-chatbot modtager dokumenter fra brugere og bruger indholdet som kontekst. Du er bekymret for, at brugere kan indsætte skjulte instruktioner i dokumenterne for at manipulere LLM'ens adfærd. Hvilken feature adresserer dette?

A) Azure AI Search Semantic Ranker — screener dokumenter for manipulerende indhold
B) Azure AI Content Safety Prompt Shields — detekterer indirect prompt injection
C) RAI Dashboard Error Analysis — identificerer uventede input-mønstre
D) Azure ML Responsible AI Counterfactuals — beregner alternative input-scenarier

---

**Spørgsmål 9 (SHAP):**
En salgsmedarbejder spørger: *"Min model forudsagde at denne kunde vil churn — hvilke specifikke faktorer for netop DENNE kunde drev forudsigelsen?"* Hvilken RAI Dashboard-komponent og forklaringstype bruger du?

A) Error Analysis med global kohort-analyse
B) Explanations med lokal feature importance (SHAP for én observation)
C) Causal Analysis med interventions-estimering
D) Counterfactuals med minimal input-ændring

---

**Spørgsmål 10 (Chunking):**
Du indexerer lange juridiske dokumenter (gennemsnit 50 sider) til en RAG-pipeline. Du oplever at retrievede chunks mangler kontekst, fordi vigtig baggrundsinformation er i starten af dokumentet, men det relevante afsnit er på side 30. Hvilken chunking-strategi afhjælper bedst dette problem?

A) Reducer chunk-størrelsen til 128 tokens for mere præcis retrieval
B) Brug fixed-size chunks med større overlap (40-50%) mellem chunks
C) Brug paragraph/section-based chunking kombineret med at inkludere et dokument-niveau summary-chunk
D) Fjern chunking helt og send hele dokumentet som kontekst

---

**Spørgsmål 11 (Indexer vs. Index):**
Hvad er den korrekte beskrivelse af forholdet mellem en **Indexer** og et **Index** i Azure AI Search?

A) Et Index definerer datakilde-forbindelsen; en Indexer definerer feltstrukturen
B) En Indexer er processen der henter, beriger og udfylder Index'et; Index'et er søgelageret
C) Indexer og Index er synonymer i Azure AI Search-terminologi
D) En Indexer er en type Skillset der opdaterer Index'et i realtid via streaming

---

**Spørgsmål 12 (On Your Data):**
Du vil bygge en prototype-chatbot der svarer på spørgsmål om dit Azure AI Search-index på under to timer, med minimal kode. Hvilken tilgang er hurtigst?

A) Byg et Prompt Flow med Python Tools til retrieve-trin og LLM Tool til generate-trin
B) Brug Azure OpenAI "On Your Data" med Azure AI Search som datakilde
C) Skriv en Azure Function der kalder Search API og OpenAI API manuelt
D) Fine-tune GPT-4o på dit index-indhold og deploy via Serverless API

**Dine svar på Eksamensquiz:**

1. 

B

2. 

B, vector search fields

3. 

B

4. 

C, OCR trækker tekst ud fra billedet. En forudsætning for tekstuel søgning.

5. 

De er inkompabitle. De mapper tekst ind i vektorrummet forskelligt. 

6. 

Error Analysis, kohorte-analyse. 

7. 

8. 

9. 

10. 

11. 

12. 

---
## 17. Facit med forklaringer
---

Tjek dine svar herunder. **Læs forklaringerne** — de er formuleret som eksamenshuske-regler.

---

**1. Svar: B** — `searchable=false, filterable=true, sortable=true`
> Attributterne er uafhængige. `filterable` giver `$filter`-support. `sortable` giver `$orderby`-support. `searchable` aktiverer fuld-tekst analyse — slå det fra for numeriske felter der ikke skal søges tekstmæssigt.

---

**2. Svar: B** — Vector Search kræver `Collection(Edm.Single)`-felt med embedding-vektorer
> Semantic Search re-ranker eksisterende resultater — den *finder* ikke nye semantisk relevante dokumenter. Kun Vector Search finder dokumenter baseret på semantisk lighed via embedding-nærhed. Kræver pre-genererede vektorer i index.

---

**3. Svar: B** — RAG med Azure AI Search + Azure OpenAI
> Fine-tuning er til at ændre modeladfærd, ikke til at tilføje viden. Månedlig fine-tuning af GPT-4o er ekstremt dyrt. RAG med indexopdatering er billig, hurtig og giver kildehenvisninger. Til faktabaseret, opdaterbar viden: brug altid RAG.

---

**4. Svar: C** — OCR-skill
> Scannede dokumenter er billeder — teksten er ikke maskine-læsbar. OCR (Optical Character Recognition) konverterer billedindhold til tekst, som de øvrige skills og search-engine derefter kan arbejde med.

---

**5. Svar: B** — Vektorerne er inkompatible
> Selvom begge modeller producerer 1536-dimensionelle vektorer, er vektorrummene *ikke* kompatible — de er trænet forskelligt. Cosine similarity på tværs af modeller er meningsløs. Du skal re-indeksere alle dokumenter med den nye model.

---

**6. Svar: C** — Error Analysis
> Error Analysis er komponenten der identificerer hvilke **datakohorteer** (subgrupper) modellen fejler på via et fejltræ og heatmap. Den er designet præcis til dette scenarie: "performer modellen dårligere for en bestemt gruppe?"

---

**7. Svar: B** — Fairness
> Fairness-princippet handler om uretfærdig behandling baseret på demografiske karakteristika. Selvom Inclusiveness og Reliability også er relevante, er Fairness det primære princip der krænkes når systemet systematisk discriminerer baseret på hudfarve.

---

**8. Svar: B** — Azure AI Content Safety Prompt Shields
> Prompt Shields er specifikt designet til at detektere *indirect prompt injection* — skjulte instruktioner i dokumenter/kontekst der forsøger at manipulere LLM'ens adfærd. Det er en separat tjeneste fra RAI Dashboard.

---

**9. Svar: B** — Explanations med lokal feature importance
> Global importance viser hvad der generelt driver modellen. Lokal importance (SHAP for én observation) viser hvad der drev forudsigelsen for *netop denne kunde*. Det er det salgsmedarbejderen beder om.

---

**10. Svar: C** — Section-based chunking + dokument-niveau summary-chunk
> Problemet er at kontekst fra side 1 er nødvendig for at forstå side 30. Løsningen er at chunke på afsnits-/sektionsniveau for at bevare lokal sammenhæng, og tilføje et summary-chunk per dokument der kan retrieves som baggrundskontekst. Større overlap (svar B) hjælper ikke ved 30-siders afstand.

---

**11. Svar: B** — Indexer er processen, Index er lageret
> En Indexer henter data fra en datakilde, sender det igennem et Skillset, og gemmer resultater i et Index. Index'et er søgelageret der eksponeres til søgeforespørgsler. De er separate ressourcer med separate roller.

---

**12. Svar: B** — Azure OpenAI "On Your Data"
> "On Your Data" håndterer retrieve-and-augment automatisk — du angiver blot dit Search index som datakilde i API-kaldet. Det er den hurtigste vej til en prototype-RAG-chatbot. Prompt Flow giver mere kontrol men kræver mere konfiguration.

---
## Nøglepunkter til eksamen — Dag 10
---

**Azure AI Search — Arkitektur:**
- Fire komponenter: `Datakilde` → `Indexer` → `Skillset` → `Index`
- `Index` = søgelager (felter + attributter). `Indexer` = arbejdsproces der udfylder index'et
- `Skillset` = AI-berigelsespipeline tilknyttet indexeren

**Azure AI Search — Feltattributter:**
- `searchable` = fuld-tekst søgning. `filterable` = `$filter`. `sortable` = `$orderby`. `facetable` = facet-aggregering
- Attributterne er **uafhængige** — du kan have filterable men ikke searchable (f.eks. til numeriske filtre)
- `facetable` og `sortable` virker ikke på Collection-felter

**Azure AI Search — Søgetyper:**
- `Keyword`: BM25 fuld-tekst, ingen semantisk forståelse
- `Semantic`: Re-ranker keyword-resultater med sprogmodel — kræver Standard SKU
- `Vector`: Finder semantisk lignende dokumenter via embeddings — kræver `Collection(Edm.Single)` i index
- `Hybrid` = Keyword + Vector (RRF fusion) — bedste til RAG i produktion

**RAG-pattern:**
- Tre trin: **Retrieve** (søg i index) → **Augment** (byg prompt med kontekst) → **Generate** (kald LLM)
- RAG til faktabaseret, opdaterbar viden. Fine-tuning til adfærdsændring, tone, format
- Chunking-vigtig: samme embedding-model til indexering og søgning — aldrig bland modeller
- "On Your Data": Azure OpenAI håndterer RAG-retrieval automatisk — hurtigst til prototyper

**Responsible AI — 6 principper:**
- `Fairness` = bias mod grupper
- `Reliability & Safety` = fejler i edge cases
- `Privacy & Security` = datalækage
- `Inclusiveness` = ekskluderer brugergrupper
- `Transparency` = kan ikke forklare beslutninger
- `Accountability` = ingen ansvarlig

**RAI Dashboard — 4 komponenter:**
- `Error Analysis` = fejltræ, find kohorteer med høj fejlrate
- `Explanations` = SHAP feature importance (global: alle obs., lokal: én obs.)
- `Counterfactuals` = minimal ændring for andet udfald
- `Causal Analysis` = kausale effekter af feature-ændringer (ikke kun korrelation)

**Content Safety:**
- Separat Azure AI-tjeneste — ikke en del af RAI Dashboard
- Detekterer: Hate, Violence, Sexual, Self-harm (score 0-7)
- `Prompt Shields`: direkte jailbreak + **indirect prompt injection** (skjulte instruktioner i dokumenter)
- Bruges som runtime-barriere foran (input) og bag (output) LLM'en

**Huske-regler til eksamen:**
- Embedding-modeller: SAMME model til indexering og søgning — altid
- RAG vs. Fine-tuning: dynamisk data = RAG, adfærdsændring = fine-tuning
- Error Analysis = HVILKE grupper fejler. Explanations = HVORFOR en forudsigelse. Counterfactuals = HVAD skal ændres
- Content Safety = runtime-beskyttelse. RAI Dashboard = model-analyse (offline)
- Hybrid Search = bedste til RAG (keyword + vector + valgfri semantic re-ranking)